# ALGORITHMIC TRADING FINAL PROJECT

- project abstract

In [2]:
%%html
<style>
table {float:left}
table + * {clear: both}
</style>

## 1. Project Setup

### 1.1 Trading Platform

#### Broker Selection Decision

After completing the initial exploration of forex trading platforms, several brokers were evaluated for automated trading in Germany. The criteria for selection were full regulation according to EU/BaFin standards, comprehensive Python API access, and a wide range of trading instruments. The choice fell on Interactive Brokers, which provides a tested platform for real futures contracts, has excellent Python API support (via the ib_async library), offers competitive fees and a wide instrument range, and is fully regulated in Germany.

Interactive Brokers (IBKR) is a global online brokerage firm headquartered in the United States. It provides direct access to a wide range of financial markets and instruments, including stocks, options, futures, bonds, and foreign exchange. It is known for its strong technological focus, low trading costs, and infrastructure that not only fulfills the needs of professional and institutional participants but is also suitable for retail traders.

The company was founded by Thomas Peterffy, one of the early pioneers of algorithmic trading. Starting as an architectural draftsman working on highway projects, he left this career path to trade equity options and apply his strategies to computerized trading models. He developed some of the first automated execution systems, thereby laying the groundwork for modern electronic markets. This technology-driven background is still visible today in Interactive Brokers' powerful trading platform and extensive API support.

For traders based in Europe—and especially in Germany—Interactive Brokers is a practical and reliable choice. The broker operates through regulated European entities and offers broad access to international markets. Compared to many alternatives, it complies with local financial regulations and provides competitive fees and transparent pricing. The main drawback is that it requires expensive data subscriptions for live streaming data of certain asset classes like stocks.

Features such as fully functional paper trading accounts and robust API connectivity make IBKR particularly well suited for algorithmic and systematic trading projects. Interactive Brokers combines regulatory reliability, global market access, and strong technical capabilities, making it a natural choice for quantitative and algorithmic trading workflows.

#### The GUI Requirement Challenge

The key technical challenge with Interactive Brokers is that API access requires a GUI environment, either Trader Workstation (TWS) or IB Gateway. This approach presents several problems:

- Headless servers: Cloud instances don't have display environments
- 24/7 operations: Running locally requires keeping a computer on continuously
- Remote access: Difficult to access from different locations
- Stability: Interruptions of local networks affect trading operations

The solution that aligns with this project's requirements for a fully automated trading system was to deploy IB Gateway in a Docker container.

#### Cloud Instance Requirements Analysis

For IB Gateway deployment in a Docker container, the following requirements were identified:

__Technical Requirements:__
- Linux OS (Ubuntu preferred for compatibility)
- Sufficient RAM for Java process (IB Gateway is Java-based)
- Stable network connection
- Docker support

__Cost Considerations:__
- Must be more economical than running a local computer 24/7
- Predictable monthly costs
- Included bandwidth for trading data

__Geographic Considerations:__
- Low latency to Interactive Brokers servers
- European datacenter preferred (trading from Germany)

__Specifications of Selected Configuration (DigitalOcean Droplet):__
```
OS:          Ubuntu 22.04 LTS
Plan:        Basic $12/month
Resources:   2 GB RAM / 1 CPU / 50 GB SSD
Datacenter:  Frankfurt, Germany
Transfer:    2 TB included
````
Ubuntu 22.04 LTS was chosen for its long-term support, excellent Docker compatibility, and extensive documentation. Since Java processes can be memory-intensive, the 1 GB RAM option was deemed insufficient, leading to the selection of 2 GB RAM.

Frankfurt was selected as the datacenter location because, being located in Germany, a datacenter with the lowest latency from Germany and proximity to IB's European servers was needed. The $12/month tier provides reasonable costs while guaranteeing better performance and stability — much cheaper than running a computer or laptop at home 24/7.

### 1.2 Initial Server Setup

#### Phase 1: Droplet Creation

After setting up a DigitalOcean account, a droplet needs to be created with the previously described configuration. This involves:

- Selecting Ubuntu 22.04 LTS image
- Choosing Basic plan with 2 GB RAM
- Selecting Frankfurt datacenter
- Adding SSH public key for authentication
- Naming droplet: ib-gateway-server

To connect to the droplet from a local machine, an SSH key needs to be generated first:
```
# Generate SSH key on Mac terminal
ssh-keygen -t ed25519 -C "your_email@example.com"
   
# Display public key
cat ~/.ssh/id_ed25519.pub
```

Ed25519 is a modern, high-security digital signature algorithm based on elliptic curve cryptography, specifically designed for speed, safety, and resistance to common implementation flaws. It's faster and more secure than older RSA keys, making it the recommended choice for SSH authentication.

__Note:__ In the following descriptions, the actual IP address of the cloud instance (provided by DigitalOcean) will be replaced with XXX.XXX.XXX.XX for security reasons.

#### Phase 2: Initial Server Configuration

Once the droplet is created and accessible, the server needs to be secured and prepared for Docker deployment.

__1. Connecting to droplet via SSH:__
```
# Connect to new droplet
ssh root@XXX.XXX.XXX.XX
```

__2. Updating the operating system:__
```
# Update package lists
apt update
   
# Upgrade installed packages
apt upgrade -y
   
# Install useful tools
apt install -y curl wget vim net-tools
```

__3. Setting up a firewall:__
```
# Install firewall
apt install -y ufw
   
# Allow SSH (critical - don't lock yourself out!)
ufw allow 22/tcp
   
# Enable firewall
ufw enable
   
# Check status
ufw status
```
Expected output:
```
Status: active

To                         Action      From
--                         ------      ----
22/tcp                     ALLOW       Anywhere
````
Initially, restricting SSH access to a specific IP address was considered. However, with a dynamic home IP, keeping SSH open with key authentication proved to be a more practical solution while still maintaining strong security.

#### Phase 3: Docker Installation

Docker is required to run IB Gateway in a containerized environment, providing isolation and easy management of the application.

__1. Installing Docker:__
```
# Download Docker installation script
curl -fsSL https://get.docker.com -o get-docker.sh
   
# Run installation script
sh get-docker.sh
   
# Verify installation
docker --version
```
Expected output: `Docker version 24.0.x, build xxxxxxx`

__2. Testing Docker__
```
# Run test container
docker run hello-world
```
If Docker is properly installed, this command will download a test image and display a "Hello from Docker!" message confirming that the installation is working correctly.

### 1.3 IB Gateway Container Deployment

#### Phase 1: Docker Image Selection

After researching available IB Gateway Docker images, `ghcr.io/unusualalpha/ib-gateway:latest` was found to be the most suitable and robust option. The selected image:

- Is the most popular and actively maintained
- Handles xvfb (virtual display) automatically
- Includes IBC (Interactive Brokers Controller) for automation
- Is well-documented and widely used in the community
- Has regular updates to match IB Gateway releases

#### Phase 2: Credentials Configuration

__1. Creating configuration directory:__
```
mkdir -p ~/ib-config
cd ~/ib-config
```

__2. Creating credentials file:__
```
nano ib-credentials.env
```
File contents of `ib-credentials.env`:
```
TWS_USERID=your_ib_username
TWS_PASSWORD=your_ib_password
TRADING_MODE=paper
VNC_SERVER_PASSWORD=your_vnc_password
```
Explanation of parameters:
- TWS_USERID: Interactive Brokers username
- TWS_PASSWORD: Interactive Brokers password
- TRADING_MODE: paper for paper trading, live for real trading
- VNC_SERVER_PASSWORD: Password for VNC remote desktop access

__3. Securing credentials file:__
```
# Set restrictive permissions (owner read/write only)
chmod 600 ib-credentials.env
   
# Verify permissions
ls -la ib-credentials.env
# Should show: -rw------- 1 root root
```

#### Phase 3: Container Deployment

__1. Pulling the Docker image:__
```
docker pull ghcr.io/unusualalpha/ib-gateway:latest
```

__2. Running the IB Gateway container:__
```
docker run -d \
     --name ib-gateway \
     --restart unless-stopped \
     -p 4001:4001 \
     -p 4002:4002 \
     -p 5900:5900 \
     --env-file ~/ib-config/ib-credentials.env \
     ghcr.io/unusualalpha/ib-gateway:latest
```
Breakdown of Docker commands:
- `-d`: Run in detached mode (background)
- `--name ib-gateway`: Name the container for easy reference
- `--restart unless-stopped`: Auto-restart on failures or system reboot
- `-p 4001:4001`: Paper trading API port
- `-p 4002:4002`: Live trading API port
- `-p 5900:5900`: VNC port for GUI access
- `--env-file`: Load credentials from secure file

__3. Verifying container is running:__
```
# Check container status
docker ps
   
# Should show ib-gateway with STATUS "Up"
```

__4. Monitoring startup logs:__
```
# View logs
docker logs ib-gateway
   
# Follow logs in real-time
docker logs -f ib-gateway
```
Expected log messages:

- "IBC: Starting Gateway"
- "IBC: Detected frame entitled: IBKR Gateway"
- "IBC: Login attempt"
- "IBC: Login has completed"

#### Phase 4: Initial Configuration via VNC

At this stage of the configuration process, first-time paper trading accounts require the configuration of certain API settings through the GUI interface. To accomplish this, it is necessary to connect to the IB Gateway instance running on the droplet from a local computer (in this case, a Mac) via VNC.

__1. Connecting to VNC from local Mac:__
- Opening Finder
- Pressing Cmd+K
- Entering server address: vnc://XXX.XXX.XXX.XX:5900
- Entering VNC password (from credentials file)

__2. Verify API Settings:__
Once connected, navigate to: Configure → Settings → API → Settings
- Verify the following settings:
  - Socket port: 4002 (for live) or 4001 (for paper)
  - Trusted IPs includes 127.0.0.1
  - "Read-Only API" is _unchecked_
  - "Allow connections from localhost only" is _unchecked_

#### Phase 5: Connection Architecture and SSH Tunnel

__The IPv4/IPv6 Challenge__  
Initial attempts to connect directly from Mac to droplet failed due to IPv4/IPv6 networking issues between Docker, the host OS, and the remote client.

__Technical Details:__
- IB Gateway Java process was binding to IPv6 (:::4002)
- Mac's ib_async was attempting IPv4 connection
- Docker bridge networking added complexity
- Firewall considerations created additional challenges

__SSH Tunnel Solution__  
After researching solutions, SSH tunneling emerged as the recommended approach for remote IB Gateway access. This method not only bypasses networking issues but also provides enhanced security compared to direct connections.

__How It Works:__
```
Local Mac:4002 → SSH Tunnel → Server:22 → Server:localhost:4002 → IB Gateway
```

__Implementation:__

__1. Creating an SSH tunnel on local computer__
```
# -4 forces IPv4
# -L creates local port forwarding
ssh -4 -L 4002:127.0.0.1:4002 root@XXX.XXX.XXX.XX
```

The terminal window where the tunnel is running needs to be kept open. Closing it terminates the tunnel.

__2. Connecting to the tunnel from a Python script:__
```
# Connect to localhost (tunneled to server)
from ib_async import *

# Create IB connection object
ib = IB()
ib.connect('127.0.0.1', 4002, clientId=1, timeout=20)

# Verify connection
print(f"Connected: {ib.isConnected()}")
```
Expected output:
```
Connected: True
```

__Security Benefits of SSH Tunnel:__
The SSH tunnel approach provides security advantages that make it superior to direct port exposure. First, all communication between the local machine and the server is encrypted through the SSH protocol, protecting sensitive trading data and credentials from interception. This encryption applies to the entire data stream, including API commands and market data.

Second, the tunnel eliminates the need to expose IB Gateway ports (4001, 4002) directly to the internet. Instead, only the SSH port (22) needs to be accessible, significantly reducing the attack surface. Any attempt to access the trading API must first authenticate through SSH using key-based authentication, adding an additional security layer that port-based connections lack.

Third, this architecture provides flexibility regardless of network configuration. Whether dealing with IPv4/IPv6 issues, NAT traversal, or Docker networking complexities, the SSH tunnel creates a reliable connection path. This is why SSH tunneling is considered standard practice among professional trading firms for remote gateway access—it combines robust security with practical reliability.

__3. Final Firewall Configuration:__
```
# Only SSH port exposed
ufw status
```
Expected output:
```
Status: active

To                         Action      From
--                         ------      ----
22/tcp                     ALLOW       Anywhere
22/tcp (v6)                ALLOW       Anywhere (v6)
```

#### Conclusion

This deployment successfully established a production-grade infrastructure for automated trading. The IB Gateway now runs continuously on the DigitalOcean droplet, independent of any local computer, providing true 24/7 operation. The multi-layered security architecture—combining SSH key authentication, firewall restrictions, and encrypted tunneling—ensures that trading operations are protected while remaining accessible from any location with SSH access.

The financial benefits are substantial: at \\$12 per month, the cloud-based solution costs significantly less than running a local computer continuously, which would consume \\$50-100 monthly in electricity and wear. Docker's automatic restart capability ensures the gateway remains operational even after system updates or unexpected failures, eliminating the need for manual intervention.

Perhaps most importantly, this architecture is not merely a proof-of-concept but represents the same approach used by professional trading firms. The combination of containerization, secure remote access, and reliable cloud infrastructure creates a foundation suitable for both development and production trading environments. The system is now ready to support automated trading strategies with the reliability and security that real-world trading demands.

## 2. Strategic Decisions

### 2.1 Choice of Instrument

As the trading instrument, the EUR/USD currency pair is chosen. The focus is on direct foreign exchange trading rather than CFDs, because this allows to put a greater emphasis on macroeconomic factors and geopolitical events that have a direct influence on currency values. Given the current political situation between Europe and the United States, this currency pair is expected to show a high level of unpredictability and volatility, making it particularly interesting to trade in the near future. From a practical perspective, forex trading is available immediately within my Interactive Brokers' account setup and does not require costly subscriptions for live market data.

### 2.2 Trading Data

For the backtesting task, historical data for the EUR/USD currency pair is obtained from Interactive Brokers with the EClient.reqHistoricalData API method.

***[more text needs to be added explaining the choice of different timeframes]***

### 2.3 Simple Moving Average Selection for Multi-Timeframe Strategy

#### Trading Style Classification

This project employs a multi-timeframe approach that spans two distinct trading styles: __day trading__ and __swing trading__. Day trading involves opening and closing positions within the same trading day (or within hours), capitalizing on intraday price movements. Swing trading, by contrast, holds positions for multiple days to weeks, aiming to capture larger price swings driven by medium-term trends (Elder, 1993; Forexearlywarning, n.d.).

The distinction between these styles is critical because each requires different analytical tools, position sizing, and—most importantly for this project — different moving average periods to filter market noise and identify meaningful trends.

#### Timeframe Classifications

This strategy implements three distinct timeframes, each aligned with a specific trading style:

| Timeframe | Trading Style | Expected Hold Time |
|-----------|---------------|-------------------|
| 5-minute | Day trading (scalping/intraday) | Minutes to hours |
| 4-hour | Day trading | Hours (same day) |
| Daily | Swing trading | Days to weeks |  

__Note on scalping:__ Scalping is an ultra-short-term subset of day trading where traders execute dozens or even hundreds of trades per day, holding positions for seconds to minutes, targeting small price movements of just a few pips (Dukascopy Bank, 2025; FXTM, n.d.). The 5-minute timeframe falls into this category when positions are closed within minutes.

#### Why Different Timeframes Require Different MA Periods

Moving averages smooth price data by calculating the average closing price over a specified number of periods. The "period" is relative to the chart timeframe: on a 5-minute chart, a 20-period SMA represents 100 minutes of price data, while on a daily chart, it represents 20 trading days (approximately one month).

Different timeframes exhibit different levels of price noise and trend characteristics:

- __5-minute charts__ contain significant intraday volatility and "noise" (random price fluctuations unrelated to meaningful trends)
- __4-hour charts__ filter out some intraday noise while still capturing same-day price movements
- __Daily charts__ smooth out intraday volatility entirely, revealing only sustained multi-day trends

Consequently, the optimal MA periods must be calibrated to each timeframe to balance two competing objectives: (1) responsiveness to genuine price changes, and (2) filtration of meaningless noise.

#### Literature-Backed SMA Periods by Timeframe

__1. Five-Minute Chart (Day Trading/Scalping)__

__Selected Periods: SMA 20 / SMA 50__

__Rationale and Citations:__
- Empirical testing by Forex.in.rs (2022) across 72 combinations on six major USD forex pairs (EUR/USD, GBP/USD, USD/JPY, AUD/USD, USD/CAD, NZD/USD) over 10 years identified the __20/50 SMA combination as optimal for 5-minute charts__
- FXOpen (2025) recommends the 20-period moving average as best suited for 5-minute forex scalping strategies
- Alternative combinations documented in practitioner literature include 10/50 and 5/8/13 (Fibonacci-based), but 20/50 provides the most robust empirical support

__References:__
- Igor (2022). *The Best Moving Average for 5 Min Chart.* Forex.in.rs. Retrieved from https://www.forex.in.rs/moving-average-for-5-min-chart/
- FXOpen (2025). *Three Working 5-Minute Trading Strategies.* Market Pulse. Retrieved from https://fxopen.com/blog/en/three-working-5-minute-trading-strategies/

__2. Four-Hour Chart (Day Trading)__

__Selected Periods: SMA 20 / SMA 50__

__Rationale and Citations:__
- The 4-hour timeframe is widely documented as a swing trading timeframe in academic and practitioner literature, but is also extensively used by day traders seeking to reduce intraday noise while maintaining same-day position management (TopBrokers, 2023; VectorVest, 2025)
- Multiple trading education sources identify __20/50 SMA as the standard combination for 4-hour charts__, with the 200 SMA often added as a long-term trend filter (Mind Math Money, 2025)
- Rayner Teo, a widely-cited technical analyst, states: "I stick to 20, 50 and 200ma and use it across all timeframes" (Teo, 2024)

__References:__
- TopBrokers (2023). *4 Hour Chart Trading Strategy Forex.* Retrieved from https://topbrokers.com/forex-strategies/4-hour-trading-strategy/
- VectorVest (2025). *What is the Best Time Frame for Swing Trading?* Retrieved from https://www.vectorvest.com/blog/swing-trading/what-is-the-best-time-frame-for-swing-trading/
- Teo, R. (2024). *The Moving Average Indicator Guide.* Trading with Rayner. Retrieved from https://www.tradingwithrayner.com/moving-average-indicator-strategy/

__3. Daily Chart (Swing Trading)__

__Selected Periods: SMA 50 / SMA 200__

__Rationale and Citations:__
- The __50/200 SMA crossover__ is one of the most widely followed technical indicators in financial markets, commonly known as the "Golden Cross" (bullish signal when 50 SMA crosses above 200 SMA) and "Death Cross" (bearish signal when it crosses below)
- Murphy (1999) in *Technical Analysis of the Financial Markets* — considered the definitive reference text on technical analysis—states: "The most commonly watched averages in stocks are 50 and 200 days" (p. 206)
- Elder (1993) in *Trading for a Living* discusses the use of moving averages for swing and position trading, emphasizing their role in trend identification across daily and weekly timeframes
- The 50/200 combination is ubiquitous in institutional trading and appears as default settings in most professional charting platforms (TradingView, Bloomberg Terminal, MetaTrader)

__References:__
- Murphy, J. J. (1999). *Technical Analysis of the Financial Markets: A Comprehensive Guide to Trading Methods and Applications.* New York Institute of Finance. (p. 206)
- Elder, A. (1993). *Trading for a Living: Psychology, Trading Tactics, Money Management.* John Wiley & Sons.

#### Summary Framework

The selected SMA periods follow established conventions in technical analysis literature and empirical forex trading research:

| Timeframe | Fast SMA | Slow SMA | Ratio | Primary Reference |
|-----------|----------|----------|-------|-------------------|
| __5-min__ | 20 | 50 | 1:2.5 | Forex.in.rs (2022) |
| __4-hour__ | 20 | 50 | 1:2.5 | Teo (2024); TopBrokers (2023) |
| __Daily__ | 50 | 200 | 1:4 | Murphy (1999); Elder (1993) |

Note that the __20/50 ratio (1:2.5)__ is maintained across both intraday timeframes, providing consistency in the relative spacing between fast and slow averages. This ratio has been empirically validated through extensive backtesting on forex pairs. The daily chart employs the industry-standard __50/200 ratio (1:4)__, which has decades of documented use in both academic literature and professional trading practice.

#### Considerations for Implementation

While these SMA periods are well-supported by literature, it is important to acknowledge a limitation inherent to all moving average crossover strategies: __the tendency to generate false signals in ranging (sideways) markets__, commonly called "whipsawing." This occurs when price oscillates around the moving averages without establishing a clear trend, resulting in frequent crossover signals that quickly reverse. Academic research has demonstrated that transaction costs from excessive trading can erode or eliminate profitability in such conditions (Glabadanidis et al., 2023; Marshall et al., 2022).

To address this limitation, the strategy incorporates additional confirmation indicators (RSI and Momentum), discussed in the following section.

### 2.4 Confirmation Filters - RSI and Momentum Parameter Selection

#### The Whipsawing Problem and Multi-Indicator Solution

As noted in the previous section, simple moving average crossover strategies are vulnerable to generating false signals in ranging or sideways markets — a phenomenon known as "whipsawing." This occurs when price oscillates around the moving averages without establishing a clear directional trend, producing frequent crossover signals that quickly reverse. The result is excessive trading activity that erodes profitability through accumulated transaction costs.

Academic research has consistently demonstrated that technical trading strategies, while showing promise in trending markets, often fail when realistic transaction costs are included (Glabadanidis et al., 2023). Studies show returns from MA strategies are "very sensitive to the introduction of moderate transaction costs," with performance degrading significantly when tested out-of-sample (Marshall et al., 2022). The core problem is that MA crossovers alone cannot distinguish between genuine trend changes and temporary price fluctuations.

To address this limitation, this strategy employs a __multi-indicator confirmation approach__ that requires alignment across three separate technical indicators before generating trade signals. This filtering methodology is well-established in practitioner literature: "RSI is very difficult to use without using other technical analysis tools because the indicator tends to produce false signals when market conditions change. Thus, it is advisable not to rely solely on RSI but to use it as a complementary tool" (FTMO Academy, 2025). Similarly, multiple sources emphasize that "momentum indicators should not be used alone and should be combined with other forms of analysis for more informed decision-making" (OANDA, 2024; Capital.com, n.d.).

### 2.5 Technical Indicators: RSI and Momentum

#### Relative Strength Index (RSI)

The Relative Strength Index (RSI) is a momentum oscillator developed by J. Welles Wilder in 1978 that measures the speed and magnitude of recent price changes. The indicator oscillates between 0 and 100, with traditional interpretation suggesting values above 70 indicate overbought conditions (potential downward reversal) and values below 30 indicate oversold conditions (potential upward reversal).

__Conceptual Role in the Strategy:__

RSI serves as a __momentum confirmation filter__ in this strategy. Rather than using RSI's traditional overbought/oversold signals as standalone trade triggers, the strategy requires RSI to confirm that sufficient momentum exists in the direction indicated by the MA crossover. This approach aligns with established best practices: "The indicator is best used to confirm a price action trading strategy, instead of using it to find trade signals on its own" (StockEdge, 2025).

__Key Consideration - False Signals in Trending Markets:__

An important limitation of RSI is that it can remain in overbought or oversold territory for extended periods during strong trends. As OANDA (2024) notes: "In trending market conditions, the RSI can stay in the overbought or oversold zones for extended periods... taking a trade while the RSI is in these zones could result in a 'false signal' due to trading against the trend." This is precisely why RSI must be used in conjunction with trend-following indicators (MA crossovers) rather than in isolation.

#### Momentum Indicator

The Momentum indicator measures the rate-of-change (ROC) in an asset's price over a specified lookback period. It is calculated by subtracting the closing price n periods ago from the current closing price. A positive momentum value indicates upward price acceleration, while a negative value indicates downward acceleration.

__Conceptual Role in the Strategy:__

Momentum serves as a __directional strength validator__ that confirms whether price movement has genuine force behind it. EBC Financial Group (2025) explains: "Momentum confirms trend strength when above or below zero with a clear slope, working best with price and volume to filter false signals and whipsaws." By requiring positive momentum for long signals and negative momentum for short signals, the strategy filters out weak or ambiguous price movements that are more likely to reverse.

The combination of RSI (momentum magnitude) and Momentum (directional strength) provides complementary confirmation: RSI ensures sufficient momentum exists, while Momentum validates the direction. As Charles Schwab notes: "While RSI, MACD, and ADX values can help gauge a trend's strength, they may be more informative when combined with other confirming signals" (Schwab, n.d.).

#### Parameter Selection Methodology

Rather than using default or arbitrary indicator parameters, this project adopts a __systematic parameter optimization approach__ through backtesting. This methodology aligns with quantitative trading best practices and ensures parameters are calibrated to the specific characteristics of EUR/USD forex data.

#### Approach to Parameter Selection

1. __RSI Period and Thresholds:__
   - __Standard period:__ 14 periods (Wilder's original specification, widely used across all timeframes)
   - __Threshold levels:__ Will be systematically tested across a range (e.g., 30/70, 40/60, 20/80)
   - __Rationale:__ "For more volatile assets like foreign exchange, traders may raise the thresholds to 80/20 to try and filter out false signals" (Admiral Markets, 2021)  
&nbsp;  
2. __Momentum Lookback Period:__
   - __Time-proportional scaling:__ Lookback periods will be scaled relative to the SMA periods for each timeframe
   - __Example logic:__ If using SMA 20/50 on 5-minute charts, momentum might use a 10-period lookback (half the fast SMA)
   - __Testing range:__ Multiple lookback values will be tested to find optimal performance  
&nbsp;  
3. __Optimization Constraints:__
   - Parameters will be optimized on a training dataset and validated on out-of-sample data to avoid overfitting
   - The strategy prioritizes parameter sets that show consistent performance across different market conditions
   - Transaction costs will be explicitly included in all backtesting to ensure real-world viability

#### Strategic Framework Summary

The multi-indicator approach employs three confirmation layers:

1. __Moving Average Crossover__ → Identifies trend direction
2. __RSI Filter__ → Confirms sufficient momentum magnitude
3. __Momentum Filter__ → Validates directional strength

A trade signal is generated only when all three indicators align, significantly reducing false signals compared to MA crossovers alone. This conservative approach accepts fewer trade opportunities in exchange for higher-quality signals and lower transaction costs.

#### Transition to Implementation

With the conceptual framework established, the next phase involves implementing this multi-indicator strategy in Python and conducting systematic backtesting across the three timeframes (5-minute, 4-hour, and daily). The backtesting process will:

- Determine optimal RSI thresholds and Momentum lookback periods for each timeframe
- Measure strategy performance using risk-adjusted metrics (Sharpe ratio, maximum drawdown, win rate)
- Compare multi-timeframe performance to assess which trading style (day trading vs swing trading) proves most effective for this approach
- Validate results on out-of-sample data to ensure robustness

### 2.6 Backtesting Methodology & Performance Metrics

Before implementing the multi-indicator strategy, it is essential to establish a rigorous backtesting framework that ensures results are both meaningful and resistant to statistical artifacts. This section outlines the methodological approach for evaluating strategy performance, including data partitioning, transaction cost modeling, and the specific metrics used to assess profitability and risk.

#### Data Partitioning Strategy

To evaluate strategy performance while minimizing overfitting bias, this project employs a __time-based train-test split__ rather than random sampling. Time-based splitting is critical in financial time series because:

1. __Temporal ordering matters__: Future data must not leak into training
2. __Market regime changes__: Strategies must prove robust across different market conditions
3. __Realistic simulation__: Traders only have access to past data when making decisions

__Partitioning Approach:__

- __Training Period (70%)__: Used for parameter optimization and strategy development
- __Testing Period (30%)__: Reserved exclusively for out-of-sample validation
- __No look-ahead bias__: All indicators and signals calculated using only data available at that point in time

__Example for 2024 daily data:__
```
Training:   January 1, 2024 → August 31, 2024 (~8 months)
Testing:    September 1, 2024 → December 31, 2024 (~4 months)
```

This approach follows established quantitative finance practices: parameters are tuned on the training set, and final performance evaluation occurs on the held-out test set. A strategy that performs well in-sample but poorly out-of-sample suggests overfitting to historical noise rather than genuine predictive power.

#### Transaction Cost Modeling

Academic research consistently demonstrates that technical trading strategies often appear profitable in backtests but fail in live trading due to underestimated transaction costs (Marshall et al., 2022; Glabadanidis et al., 2023). This project explicitly models realistic trading costs to ensure viability.

__Cost Components:__

1. __Spread__: The difference between bid and ask prices
   - EUR/USD typical spread: ~0.0001 (1 pip)
   - As percentage of price: approximately 0.0085% per side  
&nbsp;  
2. __Commission__: Broker fees per transaction
   - Interactive Brokers forex commission: minimal for retail sizes
   - Included in cost calculations where applicable  
&nbsp;  
3. __Slippage__: Difference between expected and actual execution price
   - Conservatively estimated based on order size and market liquidity
   - More significant on 5-minute bars than daily bars


__Position Change Scenarios:__

- __FLAT → LONG or SHORT__: One-way cost (entry only)
- __LONG ↔ SHORT__: Two-way cost (exit previous position + enter opposite)
- __FLAT → FLAT__: No cost (no trading activity)

Every position change in the backtest incurs the appropriate transaction cost, ensuring profitability estimates reflect realistic trading conditions. This conservative approach helps avoid the common pitfall where strategies appear profitable on paper but lose money in practice.

#### Position Sizing Approach

To maintain consistency across timeframes and simplify performance comparison, this project employs __fixed capital allocation__ rather than position scaling:

- __Capital allocation__: Fixed amount per trade (e.g., $10,000 notional)
- __Leverage__: Not used (1:1 position sizing)
- __Reinvestment__: Profits/losses accumulate but do not compound into larger positions

This approach ensures:
- Consistent risk exposure across all trades
- Comparable results between 5-minute, 4-hour, and daily timeframes
- Simplified interpretation of percentage returns

While more sophisticated position sizing methods exist (e.g., Kelly Criterion, volatility-based scaling), fixed capital allocation provides a clear baseline for strategy evaluation and can be enhanced in future iterations.


#### Performance Metrics

Strategy performance will be evaluated using a comprehensive suite of risk-adjusted metrics that capture different dimensions of profitability and risk:

__Primary Metrics:__

1. __Sharpe Ratio__
   - Formula: (Mean Return - Risk-Free Rate) / Standard Deviation of Returns
   - Interpretation: Risk-adjusted return; higher is better (>1.0 considered good, >2.0 excellent)
   - Why it matters: Accounts for volatility; a high return with extreme volatility is less desirable than moderate return with low volatility  
&nbsp;  
2. __Maximum Drawdown (MDD)__
   - Formula: Maximum peak-to-trough decline in cumulative returns
   - Interpretation: Worst possible loss from a peak; expressed as percentage
   - Why it matters: Represents worst-case scenario that traders must psychologically and financially withstand  
&nbsp;  
3. __Win Rate__
   - Formula: (Number of Profitable Trades / Total Trades) × 100%
   - Interpretation: Percentage of trades that were profitable
   - Why it matters: Indicates strategy consistency; however, a low win rate can still be profitable if winners are much larger than losers  
<br/>

__Secondary Metrics:__  

4. __Total Return__
   - Cumulative percentage gain/loss over the entire test period
   - Useful for absolute performance comparison  
&nbsp;  
5. __Average Trade Duration__
   - Mean holding period per position
   - Helps characterize trading style (scalping vs. swing trading)  
&nbsp;  
6. __Profit Factor__
   - Ratio of gross profits to gross losses
   - Values >1.0 indicate profitability; >1.5 considered strong  
&nbsp;  
7. __Number of Trades__
   - Total trades executed during test period
   - Important for assessing statistical significance and transaction cost impact
<br/>

__Multi-Metric Evaluation:__

No single metric tells the complete story. A strategy with high total return but massive drawdown is psychologically difficult to trade. Conversely, a strategy with excellent Sharpe ratio but only 10 trades lacks statistical robustness. The comprehensive metric suite enables holistic evaluation of strategy viability.

#### Parameter Optimization Framework

Each timeframe (5-minute, 4-hour, daily) will undergo systematic parameter optimization to identify the configuration that maximizes risk-adjusted returns while maintaining robustness:

__Optimization Process:__

1. __Parameter Grid Definition__:
   - RSI periods: [9, 14, 21]
   - RSI thresholds: [30/70, 40/60, 20/80]
   - Momentum lookback: [5, 10, 15, 20]
   - SMA periods: Fixed at literature-backed values (20/50 for 5-min and 4-hour; 50/200 for daily)  
&nbsp;  
2. __Objective Function__:
   - Primary: Maximize Sharpe Ratio
   - Constraint: Minimum 20 trades for statistical significance
   - Secondary: Consider Maximum Drawdown  
&nbsp;  
3. __Walk-Forward Validation__ (Time Permitting):
   - Test parameters on multiple rolling windows
   - Ensures strategy adapts to changing market conditions
   - More robust than single train-test split  
<br>

__Overfitting Prevention:__

Overfitting occurs when a strategy is excessively tuned to historical data, capturing noise rather than signal. To minimize this risk:

- __Limited parameter space__: Testing only 3-5 values per parameter rather than exhaustive search
- __Out-of-sample testing__: Final evaluation on completely separate test data
- __Reasonable constraints__: Parameters must make theoretical sense (e.g., RSI period shouldn't be 3 or 100)
- __Cross-timeframe validation__: Parameters that work across multiple timeframes are more likely genuine  
<br>

A robust strategy should demonstrate "parameter stability"—meaning small changes in parameters don't drastically change results. If performance is extremely sensitive to exact parameter values (e.g., RSI threshold of 32 works but 30 fails), this suggests overfitting.


#### Multi-Timeframe Comparison Methodology

The strategy will be tested identically across three timeframes, enabling direct comparison of trading style effectiveness:

| Timeframe | Data Granularity | Trading Style | Expected Trades/Year |
|-----------|-----------------|---------------|---------------------|
| 5-minute | High-frequency | Day trading/Scalping | 500-2000 |
| 4-hour | Medium-frequency | Day trading | 50-200 |
| Daily | Low-frequency | Swing trading | 20-100 |

__Comparison Dimensions:__

1. __Absolute Performance__: Which timeframe generates highest total return?
2. __Risk-Adjusted Performance__: Which timeframe has best Sharpe Ratio?
3. __Stability__: Which timeframe shows lowest drawdown and highest win rate?
4. __Practicality__: Which timeframe balances profitability with manageable trade frequency?

__Expected Findings:__

Literature suggests shorter timeframes (5-minute) often suffer from:
- Higher transaction costs relative to profit per trade
- More false signals from market noise
- Greater sensitivity to slippage

While longer timeframes (daily) typically offer:
- Better signal-to-noise ratio
- Lower relative transaction costs
- More stable trends to capture

However, these expectations must be validated empirically. The backtesting results will reveal whether the multi-indicator filtering approach successfully mitigates short-timeframe noise or whether longer holding periods prove inherently superior for this strategy.

#### Implementation Transition

With the theoretical framework, literature review, and methodological approach established, the project now transitions to implementation. The next sections document:

- __Section 3__: Historical data acquisition and preparation
- __Section 4__: Strategy implementation in Python
- __Section 5__: Parameter optimization and backtesting results
- __Section 6__: Performance analysis and multi-timeframe comparison

The backtesting methodology outlined here will be consistently applied across all three timeframes, ensuring fair comparison and meaningful conclusions about optimal trading approach for the EUR/USD multi-indicator strategy.

## 3. Data Preparation

### 3.1 Imports

### 3.2 Data Extraction

### 3.3 Data Preparation

### 3.4 Data Quality Assessment

### 3.5 Data Limitations and Considerations

## 4. Technical Indicator Calculation

### 4.1 Indicator Calculation

### 4.2 Indicator Summary

## 5. Signal Generation and Position Management

### 5.1 Understanding Signal Generation

### 5.2 Generating Trading Signals based on Indicator Combination

### 5.3 Converting Signals to Actual Positions

### 5.4 Signal Validation

## 6. Backtest Implementation

### 6.1 What Questions Will Backtesting Answer?

### 6.2 Strategy Readiness Assessment

### 6.3 Backtesting Methodology

### 6.4 Setting Backtest Parameters

### 6.5 Calculating Returns with Transaction Costs

### 6.6 Tracking Portfolio Value Over Time

### 6.7 Analyzing Individual Trades

### 6.8 Risk-Adjusted Performance Metrics

- Sharpe Ratio:
- Sortino Ratio:
- Maximum Drawdown:
- Calmar Ratio:

### 6.9 Comparing to Buy-and-Hold

### 6.10 Visualizing Strategy Performance

### 6.11 Backtest Results Summary

## 7. Implementing Live (Paper) Trading with Interactive Brokers

### 7.1 Overview

To transition from backtesting to real-time execution, live streaming market data from Interactive Brokers is required. In this section, I describe the implementation of a fully automated paper trading system for the EUR/USD currency pair using the Interactive Brokers Gateway. All trades are executed on a paper trading account; therefore, whenever “live trading” is mentioned in the following, it refers to simulated trading under real market conditions but with virtual capital.

The IB Gateway is already running on a cloud instance hosted at DigitalOcean (see 1. Project Setup). To enable live trading, the trading logic developed for the backtesting system must be deployed to the same server environment. This deployment was approached in three stages to ensure system reliability.


#### Stage 1: Local Development and Testing
The trading system got developed and tested on a local machine, connecting to the remote IB Gateway via SSH tunnel. This configuration allowed rapid iteration during development while streaming live market data and executing paper trades.

#### Stage 2: Extended Validation Run
After verifying stable operation over several hours, a 5-hour production run was conducted to validate long-term stability. This test remained local to identify any issues that might only emerge during extended operation before committing to cloud deployment.

#### Stage 3: Cloud Deployment
The final deployment stage consisted of moving the trading bot directly onto the cloud instance alongside the IB Gateway. This configuration eliminates network dependencies and provides maximum reliability for continuous automated trading.

This live trading system transitions the strategy from backtesting with historical data to real-time execution while preserving the same trading logic. At the same time, it introduces additional components required for handling live market data, order execution, and position management.

The implementation follows a modular architecture consisting of ***[six core Python modules, each handling specific aspects of the trading workflow. This separation of concerns enables easier testing and debugging and provides a flexible foundation for future extensions]***.

### 7.2 Preliminary Considerations for the Project

Several practical constraints of the Interactive Brokers account need to be taken into account when operating the live trading system. Only one position per instrument can be traded at any given time, meaning that long and short positions cannot be open simultaneously. In addition, the base currency of the trading account is Euro and is not automatically converted into USD for trading purposes.

Before each execution of the trading program, it is therefore necessary to check if a sufficient amount of USD cash balance is available to meet the minimum contract size for EUR/USD (20,000 units). Furthermore, existing open positions must be checked and, if necessary, closed or reversed before opening a new opposite position.

### 7.3 Requirements for the Live Trading Architecture

The live trading system is designed to satisfy the following functional requirements:

- Real-time data streaming using XXX bars
- Live indicator calculation (moving averages, RSI, momentum) on each new bar
- Signal generation using the same logic as for the backtest
- Order execution via market orders submitted to Interactive Brokers
- Position tracking to maintain an accurate view of open positions
- Trade logging with detailed records stored in CSV format
- Performance monitoring with tracking P&L real-time
- Safe shutdown procedure, including clean termination and graceful handling of market closures
- Automatic reconnection capability to handle network interruptions during long runs

### 7.4 Revisiting the Trading Strategy

### 7.5 System Architecture

### 7.6 Implementation Challenges and Solutions

***[maybe not necessary]***

### 7.7 Testing and Validation

## 8. Cloud Deployment

After successfully developing and testing the trading system locally, the next critical step involved deploying it to a cloud environment for continuous autonomous operation. The DigitalOcean droplet already hosted the IB Gateway running in a Docker container, providing the connection to Interactive Brokers' paper trading environment. The objective was to deploy the trading bot in a separate Docker container on the same droplet, ensuring reliable connectivity to the IB Gateway while maintaining independent operation. This deployment would eliminate the dependency on a local machine and enable true 24/7 trading capabilities.

### 8.1 Why Cloud Deployment Matters

Running the trading bot locally presented several practical limitations that constrained operational flexibility. The most significant constraint was the requirement to maintain an active connection from a personal computer — closing a laptop or experiencing internet connectivity issues would immediately terminate the trading session. For a system designed to operate continuously during market hours, this dependency on personal hardware was unacceptable.

Beyond availability concerns, executing trades from residential internet connections introduces latency and reliability issues compared to cloud infrastructure. Cloud providers offer direct network paths to broker servers, typically resulting in lower latency and more stable connections. Additionally, the operational costs favor cloud deployment: running a personal computer continuously consumes \\$50-100 monthly in electricity, while the DigitalOcean droplet costs just \\$12 per month.

These technical and economic considerations made cloud deployment not merely convenient, but essential for a production-grade automated trading system.

### 8.2 Infrastructure Architecture

The deployment architecture consists of two Docker containers running on the same DigitalOcean droplet:

- IB Gateway Container: Already deployed and running, providing API access to Interactive Brokers on port 4002
- Trading Bot Container: The newly deployed trading system that connects to IB Gateway

Both containers use Docker's `--network host` mode, which allows them to share the host's network stack. This configuration simplifies connectivity as the trading bot can connect to IB Gateway using `localhost:4002`, just as it does in the local development environment via the SSH tunnel.

This dual-container architecture provides several advantages. First, it maintains clear separation of concerns — the IB Gateway handles broker connectivity and authentication, while the trading bot focuses solely on strategy execution. Second, the architecture allows independent updates to either component without affecting the other. Finally, the shared network configuration simplifies deployment while maintaining the flexibility to run multiple trading bots if needed in the future.

The droplet's specifications — 2 GB RAM, 1 vCPU, and 50 GB SSD — proved adequate for this configuration. Resource monitoring during operation showed the IB Gateway consuming approximately 400-600 MB RAM while the trading bot used 200-300 MB, leaving sufficient headroom for system operations and potential scaling.

### 8.3 Containerization with Docker

Docker containerization serves multiple purposes in this deployment. Most fundamentally, it ensures consistency between development and production environments — the trading bot runs identically whether on a local Mac or the Linux cloud server. Containerization also provides isolation, preventing conflicts with system packages or other applications that might be running on the droplet.

The container configuration follows several important principles. The base image uses Python 3.11-slim to match the local development environment while minimizing image size. The unbuffered Python flag (`-u`) ensures logs appear immediately in Docker's output stream, critical for real-time monitoring. Volume mounts separate ephemeral container state from persistent data, allowing containers to be destroyed and recreated without losing trading logs or configuration changes.

#### Dockerfile 
The Dockerfile defines the container environment for the trading bot:

```
FROM python:3.11-slim

WORKDIR /app

# Install system dependencies for ib_insync
RUN apt-get update && apt-get install -y \
    gcc \
    && rm -rf /var/lib/apt/lists/*

# Copy requirements and install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy trading bot code
COPY trading_bot.py .
COPY config_live.py .

# Create directory for logs (will be mounted as volume)
RUN mkdir -p /app/logs

# Run the trading bot
CMD ["python", "-u", "trading_bot.py"]
```

Key design decisions:

- __Python 3.11-slim:__ Matches local development environment while minimizing image size
- __gcc installation:__ Required for compiling some Python package dependencies
- __Unbuffered Python (`-u` flag):__ Ensures logs appear in real-time without buffering
- __Logs directory:__ Created for mounting as a volume to persist data outside the container

#### requirements.txt
The dependency file specifies all Python packages needed:
```
ib_async==2.1.0
pandas>=2.0.0
numpy>=1.24.0
```
Note: The package is ib_async version 2.1.0, which is the correct package name for the ib-insync library.

#### .dockerignore:
To keep the build context clean and efficient:
```
__pycache__
*.pyc
*.pyo
*.log
logs/
.git
.gitignore
*.md
```

### 8.4 Deployment Process

The deployment process follows a methodical sequence, with each step clearly marked as executing on either the local machine or the cloud instance. This distinction is important because commands executed locally connect to the droplet via SSH, while commands on the cloud instance run directly in the droplet's environment. The process has been designed to be reproducible, enabling quick redeployment if modifications are needed or if the system needs to be moved to a different server.

#### Step 1: Prepare Local Files [LOCAL]
Before deployment, ensure all necessary files are present in the local project directory:

- `trading_bot.py` - Main trading logic
- `config_live.py` - Configuration file with IB_CLIENT_ID and other parameters
- `requirements.txt` - Python dependencies
- `Dockerfile` - Container definition
- `.dockerignore` - Build exclusions

#### Step 2: Transfer Files to Droplet [LOCAL]
Transfer all necessary files from the local machine to the DigitalOcean droplet:
```
# Ensure the target directory exists
ssh root@XXX.XXX.XXX.XX "mkdir -p /root/trading_bot"

# Transfer Python files and dependencies
scp trading_bot.py config_live.py requirements.txt root@XXX.XXX.XXX.XX:/root/trading_bot/

# Transfer Docker configuration files
scp Dockerfile .dockerignore root@XXX.XXX.XXX.XX:/root/trading_bot/
```

#### Step 3: Build Docker Image [CLOUD]
SSH into the droplet and build the Docker image:
```
# Connect to droplet
ssh root@157.230.113.17

# Navigate to project directory
cd /root/trading_bot

# Build the Docker image
docker build -t trading-bot:latest .

# Verify the image was created
docker images | grep trading-bot
```
The build process downloads the base Python image, installs dependencies, and packages the trading bot code into a Docker image. This typically takes 1-2 minutes on the first build.

#### Step 4: Run the Trading Bot Container [CLOUD]
Launch the trading bot in a Docker container with appropriate configuration:
```
docker run -d \
  --name trading-bot-2h \
  --network host \
  --restart unless-stopped \
  -v /root/trading_bot/logs:/app/logs \
  -v /root/trading_bot/config_live.py:/app/config_live.py \
  trading-bot:latest
```
Docker run flags explained:

- `-d`: Run in detached mode (background process)
- `--name trading-bot-2h`: Assign a meaningful name for easy reference
- `--network host`: Share the host's network stack (enables localhost:4002 connection to IB Gateway)
- `--restart unless-stopped`: Automatically restart the container if it crashes or after system reboot
- `-v /root/trading_bot/logs:/app/logs`: Mount logs directory to persist trading records outside the container
- `-v /root/trading_bot/config_live.py:/app/config_live.py`: Mount configuration file for easy updates without rebuilding

#### Step 5: Verify Successful Deployment [CLOUD]
Check that the container is running and connecting to IB Gateway:
```
# Check container status
docker ps | grep trading-bot

# View real-time logs to confirm connection
docker logs -f trading-bot-2h
```

Expected log output confirming successful connection:
```
Connected to IB Gateway at localhost:4002
Client ID: 753
Account: DU123456
Starting trading session...
Fetching initial market data...
```

Exit the log view with `Ctrl+C` and disconnect from the droplet:
```
exit
```

### 8.5 Confirming Connection to IB Gateway

The connection between the trading bot and IB Gateway represents the critical integration point in the deployment architecture. Without a successful connection, the trading bot cannot receive market data or execute orders. The following procedures verify that this connection is established correctly and remains stable during operation.

__Network Configuration__
Both containers (IB Gateway and trading bot) use Docker's `--network host` mode. This configuration means:

- The containers share the host's network namespace
- No port mapping or Docker bridge networking is involved
- The trading bot connects to IB Gateway using localhost:4002
- This is identical to how the local development environment connects

__Connection Parameters__
In `config_live.py`, the connection parameters are configured:
```
IB_HOST = "localhost"
IB_PORT = 4002  # Paper trading port
IB_CLIENT_ID = 753  # Unique ID for cloud bot
```

The client ID (753) is different from local development IDs to allow simultaneous connections from both local and cloud instances if needed.

#### Verification Steps
To confirm the trading bot successfully connected to IB Gateway:

__1. Check IB Gateway Status [CLOUD]__  
```
ssh root@157.230.113.17 "docker ps | grep ib-gateway"
```
This should show the IB Gateway container is running:
```
0d5f489eafaa   ghcr.io/gnzsnz/ib-gateway:latest   ...   Up 13 hours   ib-gateway
```

__2. View IB Gateway Logs [CLOUD]__  
```
ssh root@XXX.XXX.XXX.XX "docker logs ib-gateway | tail -20"
```
Look for connection acceptance messages when the trading bot starts.

__3. Check Trading Bot Connection [CLOUD]__  
```
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h | head -20"
```
Should show:
```
Connected to IB Gateway at localhost:4002
Client ID: 753
Starting market data stream...
```

### 8.6 Remote Monitoring

Remote monitoring capabilities transform the cloud deployment from a "fire and forget" system into a fully observable operation. While the trading bot runs autonomously without requiring constant attention, periodic monitoring provides confidence that the system operates correctly and enables early detection of any issues. All monitoring can be performed from a local machine without maintaining a persistent SSH connection to the droplet.

The monitoring approach follows Unix philosophy: small, composable commands that can be combined to answer specific questions about the system's state. Rather than connecting to the droplet and navigating its filesystem, all monitoring happens through SSH one-liners that return specific pieces of information. The following commands enable comprehensive monitoring from any location.

__Basic Status Checks [LOCAL]__  
```
# Check if the bot container is running
ssh root@XXX.XXX.XXX.XX "docker ps | grep trading-bot"

# Check container resource usage
ssh root@XXX.XXX.XXX.XX "docker stats --no-stream trading-bot-2h"
```

__Real-Time Log Monitoring [LOCAL]__  
```
# View live log output (follows new logs as they appear)
ssh root@157.230.113.17 "docker logs -f trading-bot-2h"
```
Use `Ctrl+C` to exit the live log view.

__Specific Information Queries [LOCAL]__  
```
# Check most recent P&L
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h 2>&1 | grep 'Cumulative P&L' | tail -1"

# Count total trades executed
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h 2>&1 | grep 'TRADE EXECUTED' | wc -l"

# View recent trading signals
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h 2>&1 | grep 'Signal:' | tail -10"

# Check for any errors
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h 2>&1 | grep -i error"

# Check time remaining (if bot has fixed duration)
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h 2>&1 | grep 'minutes remaining' | tail -1"
````

__Monitoring Session Example__  
A typical monitoring session from the local machine:
```
# Quick status check
$ ssh root@157.230.113.17 "docker ps | grep trading-bot"
abc123def456   trading-bot:latest   ...   Up 45 minutes   trading-bot-2h

# Check latest P&L
$ ssh root@157.230.113.17 "docker logs trading-bot-2h 2>&1 | grep 'Cumulative P&L' | tail -1"
2026-01-23 10:34:12 - Cumulative P&L: $-2.45 | Open P&L: $1.20

# Count trades so far
$ ssh root@157.230.113.17 "docker logs trading-bot-2h 2>&1 | grep 'TRADE EXECUTED' | wc -l"
18
```

### 8.7 Downloading Results

Once a trading session completes, the results exist only on the cloud server until explicitly downloaded. The trading bot writes all trade data to CSV files and generates comprehensive logs, but these remain in the droplet's filesystem. The following procedures retrieve these files to the local machine where they can be analyzed, archived, and incorporated into this notebook.

The download process uses scp, secure copy protocol, to transfer files over SSH. The wildcard patterns in the commands accommodate the timestamp-based naming scheme the trading bot uses, allowing downloads without knowing the exact timestamp when the session began.

__Create Local Results Directory [LOCAL]__  
```
# Create a dated results folder
mkdir -p ~/trading_results_$(date +%Y%m%d)
```

__Download All Result Files [LOCAL]__  
```
# Download CSV trade log
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trades_*.csv ~/trading_results_$(date +%Y%m%d)/

# Download full text log
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trading_bot_*.log ~/trading_results_$(date +%Y%m%d)/

# Download summary file
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trades_*_summary.txt ~/trading_results_$(date +%Y%m%d)/
```

__Alternative: Download Specific Files [LOCAL]__  
If the exact timestamp of your trading session is known:
```
# Example with specific timestamp (20260123_094904)
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trades_20260123_094904.csv ~/trading_results_20260123/
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trading_bot_20260123_094904.log ~/trading_results_20260123/
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trades_20260123_094904_summary.txt ~/trading_results_20260123/
```

### 8.8 Container Lifecycle Management

Docker containers follow a lifecycle: they are created, started, stopped, and eventually removed. Understanding this lifecycle enables effective management of trading sessions. The `--restart unless-stopped` flag used during deployment means containers automatically restart after server reboots or crashes, but can be manually stopped when needed. This provides the right balance between resilience and control.

__Stopping the Bot [CLOUD]__  
```
ssh root@XXX.XXX.XXX.XX
docker stop trading-bot-2h
exit
```

__Starting a Stopped Container [CLOUD]__  
```
ssh root@XXX.XXX.XXX.XX
docker start trading-bot-2h
exit
```

__Removing the Container [CLOUD]__  
After downloading results and verifying the data, the container can be removed:
```
ssh root@XXX.XXX.XXX.XX
docker stop trading-bot-2h
docker rm trading-bot-2h
exit
```

__Deploying a New Instance [CLOUD]__  
```
# 1. Update configuration file on droplet
ssh root@XXX.XXX.XXX.XX
nano /root/trading_bot/config_live.py
# (Make changes, e.g., RUN_DURATION = "8 h")
# Save and exit (Ctrl+X, Y, Enter)

# 2. Run new container with updated config
docker run -d \
  --name trading-bot-8h \
  --network host \
  --restart unless-stopped \
  -v /root/trading_bot/logs:/app/logs \
  -v /root/trading_bot/config_live.py:/app/config_live.py \
  trading-bot:latest

exit
```

### 8.9 Deployment Best Practices

The deployment process evolved through several iterations, each revealing insights that improved reliability and maintainability. The following practices emerged as particularly valuable and should be considered essential rather than optional for production deployments. These recommendations address common pitfalls and reflect lessons learned through actual operation.

#### 1. Version Control
Keep local copies of all deployment files and maintain version control:
```
git add Dockerfile requirements.txt trading_bot.py config_live.py
git commit -m "Deployment configuration for 2-hour test run"
```

#### 2. Configuration Management
Use volume mounts for configuration files rather than building them into the image. This allows quick parameter changes without rebuilding:
```
-v /root/trading_bot/config_live.py:/app/config_live.py
```

#### 3. Log Persistence
Always mount the logs directory to ensure trading data persists outside the container:
```
-v /root/trading_bot/logs:/app/logs
```

#### 4. Container Naming
Use descriptive container names that indicate the run parameters:
```
--name trading-bot-2h   # For 2-hour run
--name trading-bot-8h   # For 8-hour run
```

#### 5. Monitoring Scripts
Create local shell scripts for common monitoring tasks:
```
#!/bin/bash
# monitor-bot.sh
DROPLET_IP="XXX.XXX.XXX.XX"
CONTAINER="trading-bot-2h"

echo "=== Container Status ==="
ssh root@$DROPLET_IP "docker ps | grep $CONTAINER"

echo -e "\n=== Latest P&L ==="
ssh root@$DROPLET_IP "docker logs $CONTAINER 2>&1 | grep 'Cumulative P&L' | tail -1"

echo -e "\n=== Trade Count ==="
ssh root@$DROPLET_IP "docker logs $CONTAINER 2>&1 | grep 'TRADE EXECUTED' | wc -l"
```

### 8.10 Deployment Summary

The deployment of the trading system to the DigitalOcean cloud environment achieved its primary objectives while revealing the practical advantages of containerized cloud architectures. The use of Docker containerization ensured consistency between development and production environments — code that worked locally operated identically in the cloud. The `--network host` configuration simplified connectivity to the existing IB Gateway container by eliminating complex network bridging configurations.

The deployment process proved fully reproducible. Each step has been documented with specific commands, making it straightforward to replicate the deployment on different servers or to redeploy after modifications. This reproducibility extends beyond technical commands to include the architectural decisions and best practices that inform those commands.

Key achievements include:

- __Autonomous operation:__ The trading bot runs independently of any local machine, executing trades 24/7 without human intervention
- __Reliable connectivity:__ Successful connection to IB Gateway via localhost using host networking mode
- __Remote monitoring:__ Comprehensive SSH-based monitoring capabilities enable oversight without persistent connections
- __Data persistence:__ Trade logs and results are preserved outside containers through volume mounts
- __Reproducibility:__ Complete documentation of all steps and commands enables quick redeployment

The infrastructure now supports extended trading sessions and provides a foundation for future enhancements. Multiple trading bots could run simultaneously on the same droplet, each executing different strategies or trading different instruments. The modular architecture allows updating the trading strategy without modifying the IB Gateway configuration, and vice versa. Most importantly, the system demonstrates that production-grade automated trading infrastructure can be built using accessible cloud services and open-source tools.

## 9. Results from the Cloud Trading Run

### 9.1 Performance Summary

### 9.2 Lessons Learned and Future Scalability

## 10. Conclusion

### 10.1 Key Achievements

### 10.2 Strategy Performance Analysis

### 10.3 Technical Lessons Learned

### 10.4 Future Directions

### 10.5 Final Reflections

## References

Admiral Markets (2021). *Top RSI Settings for Day Trading: Parameter & Configuration Guide.* Retrieved from https://admiralmarkets.com/education/articles/forex-indicators/relative-strength-index-how-to-trade-with-an-rsi-indicator

Bailey, D. H., Borwein, J., López de Prado, M., & Zhu, Q. J. (2014). Pseudo-mathematics and financial charlatanism: The effects of backtest overfitting on out-of-sample performance. *Notices of the American Mathematical Society, 61*(5), 458-471.

Capital.com (n.d.). *Best Momentum Indicators for Traders.* Retrieved from https://capital.com/en-int/learn/technical-analysis/best-momentum-indicators

EBC Financial Group (2025). *Does the Momentum Indicator Confirm Trend Strength?* Retrieved from https://www.ebc.com/forex/momentum-indicator

Elder, A. (1993). *Trading for a Living: Psychology, Trading Tactics, Money Management.* John Wiley & Sons.

Forex.in.rs (2022). *The Best Moving Average for 5 Min Chart.* Retrieved from https://www.forex.in.rs/moving-average-for-5-min-chart/

FTMO Academy (2025). *RSI: Technical Indicator.* Retrieved from https://academy.ftmo.com/lesson/rsi-technical-indicator/

FXOpen (2025). *Three Working 5-Minute Trading Strategies.* Market Pulse. Retrieved from https://fxopen.com/blog/en/three-working-5-minute-trading-strategies/

Glabadanidis, P., et al. (2023). The predictive ability of technical trading rules: An empirical analysis. *Financial Markets and Portfolio Management.* https://doi.org/10.1007/s11408-023-00433-2

Marshall, B. R., et al. (2022). Technical trading rule profitability in currencies: It's all about momentum. *International Review of Financial Analysis,* 84. https://doi.org/10.1016/j.irfa.2022.102395

Murphy, J. J. (1999). *Technical Analysis of the Financial Markets: A Comprehensive Guide to Trading Methods and Applications.* New York Institute of Finance.

OANDA (2024). *A Complete Understanding of the RSI.* Retrieved from https://www.oanda.com/us-en/trade-tap-blog/trading-knowledge/understanding-the-relative-strength-index/

Pardo, R. (2008). *The Evaluation and Optimization of Trading Strategies* (2nd ed.). Wiley Trading.

Schwab, C. (n.d.). *3 Strength Indicators for Assessing Stock Momentum.* Retrieved from https://www.schwab.com/learn/story/3-strength-indicators-assessing-stock-momentum

StockEdge (2025). *Master Trading Skills With The Top 5 Momentum Indicators.* Retrieved from https://blog.elearnmarkets.com/top-5-momentum-indicators/

Teo, R. (2024). *The Moving Average Indicator Guide.* Trading with Rayner. Retrieved from https://www.tradingwithrayner.com/moving-average-indicator-strategy/

Wilder, J. W. (1978). *New Concepts in Technical Trading Systems.* Trend Research.